In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
import gc
import torch

# Delete any old model variables if they exist in the namespace
if 'model' in locals():
    del model
if 'optimizer' in locals():
    del optimizer

# Force garbage collection and empty the PyTorch cache
gc.collect()
torch.cuda.empty_cache()

In [3]:
import os, math
import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    AutoTokenizer,
    AutoModelForMultipleChoice,
    get_cosine_schedule_with_warmup,
)
import wandb  
import logging
logging.getLogger("wandb").setLevel(logging.ERROR)
os.environ["WANDB_SILENT"] = "true"

# ── CONFIG ────────────────────────────────────────────────
DATA_DIR        = Path("/kaggle/input/competitions/smart-mcq-solver-challenge")
MODEL_NAME      = "microsoft/deberta-v3-base"
DEVICE          = "cuda" if torch.cuda.is_available() else "cpu"
MAX_LEN         = 256
BATCH_SIZE      = 4
ACCUM_STEPS     = 4
EPOCHS          = 15
LR_BACKBONE     = 8e-5
LR_HEAD         = 1e-4
WARMUP_FRAC     = 0.1
GRAD_CLIP       = 1.0
LABEL_SMOOTHING = 0.1
USE_FP16        = DEVICE == "cuda"
OPTION_COLS     = ["A", "B", "C", "D", "E"]

# ── W&B ───────────────────────────────────────────────────
try:
    from kaggle_secrets import UserSecretsClient
    WANDB_API_KEY = UserSecretsClient().get_secret("WANDB_API_KEY")
except Exception:
    WANDB_API_KEY = os.environ.get("WANDB_API_KEY", "")

if not WANDB_API_KEY:
    raise ValueError(
        "WANDB_API_KEY not found. "
        "Go to Kaggle notebook -> Add-ons -> Secrets -> New Secret. "
        "Name: WANDB_API_KEY, Value: your key from https://wandb.ai/authorize"
    )

wandb.login(key=WANDB_API_KEY)
wandb.init(
    project="smart-mcq-solver",
    name="deberta-v3-base",  
    config=dict(
        model=MODEL_NAME, max_len=MAX_LEN,
        batch_size=BATCH_SIZE, accum_steps=ACCUM_STEPS,
        effective_batch=BATCH_SIZE * ACCUM_STEPS,
        epochs=EPOCHS,
        lr_backbone=LR_BACKBONE, lr_head=LR_HEAD,
        warmup_frac=WARMUP_FRAC, grad_clip=GRAD_CLIP,
        label_smoothing=LABEL_SMOOTHING, fp16=USE_FP16,
    ),
)

print(f"Device : {DEVICE}  |  FP16: {USE_FP16}")
print(f"Model  : {MODEL_NAME}")
print(f"Epochs : {EPOCHS}  |  Effective batch: {BATCH_SIZE * ACCUM_STEPS}")
print(f"W&B    : {wandb.run.get_url()}")

# ── MAP@3 ─────────────────────────────────────────────────
def apk(actual, predicted, k=3):
    if not actual: return 0.0
    score, hits = 0.0, 0
    for i, p in enumerate(predicted[:k]):
        if p in actual and p not in predicted[:i]:
            hits += 1
            score += hits / (i + 1)
    return score / min(len(actual), k)

def mapk(actuals, predictions, k=3):
    return np.mean([apk([a], p, k) for a, p in zip(actuals, predictions)])

# ── DATA ──────────────────────────────────────────────────
train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df  = pd.read_csv(DATA_DIR / "test.csv")
train_df = train_df.dropna(subset=["prompt"] + OPTION_COLS + ["answer"]).reset_index(drop=True)
test_df  = test_df.dropna(subset=["prompt"] + OPTION_COLS).reset_index(drop=True)
print(f"Train: {len(train_df)}  |  Test: {len(test_df)}")

# ── DATASET ───────────────────────────────────────────────
class MCQDataset(Dataset):
    def __init__(self, df, tokenizer, has_labels=True):
        self.df         = df.reset_index(drop=True)
        self.tok        = tokenizer
        self.has_labels = has_labels
        self.lmap       = {c: i for i, c in enumerate(OPTION_COLS)}

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row     = self.df.iloc[idx]
        q       = str(row["prompt"])
        choices = [str(row[c]) for c in OPTION_COLS]
        enc = self.tok(
            [q] * 5, choices,
            truncation=True, max_length=MAX_LEN,
            padding="max_length", return_tensors="pt",
        )
        item = {k: v for k, v in enc.items()}
        if self.has_labels:
            item["labels"] = torch.tensor(self.lmap[str(row["answer"])], dtype=torch.long)
        return item

# ── MODEL ─────────────────────────────────────────────────
print("\nLoading model ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME).to(DEVICE)
model     = model.float()

train_ds = MCQDataset(train_df, tokenizer, has_labels=True)
test_ds  = MCQDataset(test_df,  tokenizer, has_labels=False)
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
test_dl  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# ── OPTIMIZER ─────────────────────────────────────────────
head_params = ["classifier", "pooler"]
no_decay    = ["bias", "LayerNorm.weight"]

optimizer_groups = [
    {"params": [p for n, p in model.named_parameters()
                if not any(nd in n for nd in no_decay) and not any(hp in n for hp in head_params)],
     "lr": LR_BACKBONE, "weight_decay": 0.01},
    {"params": [p for n, p in model.named_parameters()
                if any(nd in n for nd in no_decay) and not any(hp in n for hp in head_params)],
     "lr": LR_BACKBONE, "weight_decay": 0.0},
    {"params": [p for n, p in model.named_parameters()
                if not any(nd in n for nd in no_decay) and any(hp in n for hp in head_params)],
     "lr": LR_HEAD, "weight_decay": 0.01},
    {"params": [p for n, p in model.named_parameters()
                if any(nd in n for nd in no_decay) and any(hp in n for hp in head_params)],
     "lr": LR_HEAD, "weight_decay": 0.0},
]

optimizer   = AdamW(optimizer_groups, eps=1e-6)
total_steps = (len(train_dl) // ACCUM_STEPS) * EPOCHS
scheduler   = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(WARMUP_FRAC * total_steps),
    num_training_steps=total_steps,
)
scaler    = torch.amp.GradScaler("cuda", enabled=USE_FP16)
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

# ── TRAINING ──────────────────────────────────────────────
print("\n" + "="*50)
print("Fine-tuning ...")
print("="*50)

best_loss  = float("inf")
global_step = 0

for epoch in range(EPOCHS):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    optimizer.zero_grad()

    for step, batch in enumerate(train_dl):
        input_ids      = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels         = batch["labels"].to(DEVICE)

        with torch.amp.autocast("cuda", enabled=USE_FP16):
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss    = criterion(outputs.logits, labels) / ACCUM_STEPS

        if not math.isfinite(loss.item() * ACCUM_STEPS):
            optimizer.zero_grad()
            continue

        scaler.scale(loss).backward()

        if (step + 1) % ACCUM_STEPS == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            scheduler.step()
            global_step += 1

            wandb.log({
                "train/step_loss": loss.item() * ACCUM_STEPS,
                "train/lr":        scheduler.get_last_lr()[0],
            }, step=global_step)

        total_loss += loss.item() * ACCUM_STEPS
        correct    += (outputs.logits.argmax(-1) == labels).sum().item()
        total      += labels.size(0)

        if (step + 1) % 100 == 0:
            print(f"  Ep{epoch+1} step {step+1}/{len(train_dl)}  "
                  f"loss={total_loss/(step+1):.4f}  acc={correct/total:.4f}  "
                  f"lr={scheduler.get_last_lr()[0]:.2e}")

    epoch_loss = total_loss / len(train_dl)
    epoch_acc  = correct / max(total, 1)
    print(f"\nEpoch {epoch+1}/{EPOCHS} - loss: {epoch_loss:.4f}  acc: {epoch_acc:.4f}\n")

    wandb.log({"train/epoch_loss": epoch_loss, "train/epoch_acc": epoch_acc, "epoch": epoch+1},
              step=global_step)

    if epoch_loss < best_loss:
        best_loss = epoch_loss
        torch.save(model.state_dict(), "/kaggle/working/best_model.pt")
        print(f"  Best model saved (loss={best_loss:.4f})")

# ── INFERENCE ─────────────────────────────────────────────
model.load_state_dict(torch.load("/kaggle/working/best_model.pt"))
print("\nLoaded best checkpoint for inference.")

def predict_top3(loader):
    model.eval()
    all_logits = []
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            with torch.amp.autocast("cuda", enabled=USE_FP16):
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            all_logits.append(outputs.logits.float().cpu().numpy())
    logits = np.vstack(all_logits)
    return logits, [[OPTION_COLS[i] for i in np.argsort(r)[::-1]][:3] for r in logits]

print("Evaluating on training set ...")
_, train_preds = predict_top3(DataLoader(train_ds, batch_size=BATCH_SIZE))
train_map3 = mapk(train_df["answer"].tolist(), train_preds)
print(f"Train MAP@3: {train_map3:.4f}")

wandb.log({"train/final_map3": train_map3}, step=global_step)

print("\nGenerating test predictions ...")
_, test_preds = predict_top3(test_dl)

submission = pd.DataFrame({
    "ID":         test_df["id"],
    "Prediction": [" ".join(p) for p in test_preds],
})
submission.to_csv("/kaggle/working/submission_deberta_v3.csv", index=False)
print("Saved -> /kaggle/working/submission_deberta_v3.csv")
print(submission.head(10).to_string(index=False))

wandb.finish()


Device : cuda  |  FP16: True
Model  : microsoft/deberta-v3-base
Epochs : 15  |  Effective batch: 16
W&B    : https://wandb.ai/uma-shettar-indian-institute-of-technology-madras/smart-mcq-solver/runs/2qo02qc3
Train: 2000  |  Test: 500

Loading model ...


config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForMultipleChoice LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
classifier.bias                         | MISSING    | 
pooler.dense.weight                     | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.weight                


Fine-tuning ...


model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

  Ep1 step 100/500  loss=1.6101  acc=0.2075  lr=1.07e-05
  Ep1 step 200/500  loss=1.6068  acc=0.2150  lr=2.14e-05
  Ep1 step 300/500  loss=1.6050  acc=0.2258  lr=3.21e-05
  Ep1 step 400/500  loss=1.5711  acc=0.2719  lr=4.28e-05
  Ep1 step 500/500  loss=1.5370  acc=0.3060  lr=5.35e-05

Epoch 1/15 - loss: 1.5370  acc: 0.3060

  Best model saved (loss=1.5370)
  Ep2 step 100/500  loss=1.2376  acc=0.5525  lr=6.42e-05
  Ep2 step 200/500  loss=1.1995  acc=0.5800  lr=7.49e-05
  Ep2 step 300/500  loss=1.1308  acc=0.6183  lr=8.00e-05
  Ep2 step 400/500  loss=1.0877  acc=0.6475  lr=7.99e-05
  Ep2 step 500/500  loss=1.0275  acc=0.6830  lr=7.97e-05

Epoch 2/15 - loss: 1.0275  acc: 0.6830

  Best model saved (loss=1.0275)
  Ep3 step 100/500  loss=0.6442  acc=0.9200  lr=7.95e-05
  Ep3 step 200/500  loss=0.6279  acc=0.9250  lr=7.91e-05
  Ep3 step 300/500  loss=0.6028  acc=0.9367  lr=7.87e-05
  Ep3 step 400/500  loss=0.5932  acc=0.9419  lr=7.82e-05
  Ep3 step 500/500  loss=0.5821  acc=0.9510  lr=7.76e-